# Stage 4B — Xception Baseline + Fusion Experiments

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()

SUBSET_NAME = "final"

MANIFEST_PATH = PROJECT_ROOT / "exports/final/csv" / f"{SUBSET_NAME}_manifest_export.csv"
EMOTION_PATH = PROJECT_ROOT / "datasets/emotion_annotated/metadata" / f"{SUBSET_NAME}_video_emotion_features.csv"
XCEPTION_SCORES_PATH = PROJECT_ROOT / "outputs" / "deepfakebench_scores" / "ThesisFinal_xception_video_scores.csv"

MERGED_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_merged_xception_emotion.csv"
RANDOM_SEED = 42
TEST_SIZE = 0.2

print(MANIFEST_PATH)
print(EMOTION_PATH)
print(XCEPTION_SCORES_PATH)

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
emotion_df = pd.read_csv(EMOTION_PATH)
scores_df = pd.read_csv(XCEPTION_SCORES_PATH)

print(manifest_df.shape, emotion_df.shape, scores_df.shape)
display(scores_df.head())

In [ ]:
if "label" in scores_df.columns:
    scores_df["label_from_detector"] = scores_df["label"].map({0: "real", 1: "fake"}).fillna(scores_df["label"])

scores_keep = ["video_id", "detector_score"]
if "label_from_detector" in scores_df.columns:
    scores_keep.append("label_from_detector")
if "n_frames" in scores_df.columns:
    scores_keep.append("n_frames")

scores_df = scores_df[scores_keep].copy()

df = manifest_df.merge(emotion_df, on="video_id", how="inner", suffixes=("", "_emo"))
df = df.merge(scores_df, on="video_id", how="inner")

df["y"] = df["label"].map({"real": 0, "fake": 1})

print(df.shape)
display(df.head())
df.to_csv(MERGED_OUT, index=False)
print(MERGED_OUT)

In [ ]:
def binary_metrics(y_true, y_score, thr=0.5):
    y_pred = (y_score >= thr).astype(int)
    return {
        "AUC": roc_auc_score(y_true, y_score),
        "ACC": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
    }

def evaluate_group_auc(df_in, group_col):
    rows = []
    for group_value, sub in df_in.groupby(group_col):
        if sub["y"].nunique() < 2:
            continue
        rows.append({
            group_col: group_value,
            "n": len(sub),
            "AUC": roc_auc_score(sub["y"], sub["detector_score"])
        })
    return pd.DataFrame(rows).sort_values("AUC", ascending=False)

In [ ]:
baseline_metrics = binary_metrics(df["y"].values, df["detector_score"].values)
baseline_metrics

In [ ]:
emotion_auc_df = evaluate_group_auc(df, "dominant_emotion")
display(emotion_auc_df)

df["arousal_bin"] = pd.qcut(df["mean_arousal"], q=3, labels=["low", "mid", "high"], duplicates="drop")
arousal_auc_df = evaluate_group_auc(df, "arousal_bin")
display(arousal_auc_df)

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df["y"]
)

print(train_df.shape, test_df.shape)

In [ ]:
emotion_features = [
    "mean_valence",
    "std_valence",
    "mean_arousal",
    "std_arousal",
    "max_arousal",
    "emotion_entropy",
    "transition_rate",
    "arousal_variation",
    "neutral_ratio",
]

fusion_features = ["detector_score"] + emotion_features

if "dominant_emotion" in df.columns:
    dummies_train = pd.get_dummies(train_df["dominant_emotion"], prefix="emo")
    dummies_test = pd.get_dummies(test_df["dominant_emotion"], prefix="emo")
    dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

    train_df_model = pd.concat([train_df.reset_index(drop=True), dummies_train.reset_index(drop=True)], axis=1)
    test_df_model = pd.concat([test_df.reset_index(drop=True), dummies_test.reset_index(drop=True)], axis=1)

    dummy_cols = list(dummies_train.columns)
    emotion_features = emotion_features + dummy_cols
    fusion_features = ["detector_score"] + emotion_features
else:
    train_df_model = train_df.copy()
    test_df_model = test_df.copy()

print(emotion_features)
print(fusion_features)

In [ ]:
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
])

X_train_base = train_df_model[["detector_score"]]
X_test_base = test_df_model[["detector_score"]]
y_train = train_df_model["y"]
y_test = test_df_model["y"]

baseline_model.fit(X_train_base, y_train)
baseline_model_scores = baseline_model.predict_proba(X_test_base)[:, 1]

baseline_model_metrics = binary_metrics(y_test.values, baseline_model_scores)
baseline_model_metrics

In [ ]:
emotion_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
])

X_train_emotion = train_df_model[emotion_features]
X_test_emotion = test_df_model[emotion_features]

emotion_model.fit(X_train_emotion, y_train)
emotion_model_scores = emotion_model.predict_proba(X_test_emotion)[:, 1]

emotion_model_metrics = binary_metrics(y_test.values, emotion_model_scores)
emotion_model_metrics

In [ ]:
fusion_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
])

X_train_fusion = train_df_model[fusion_features]
X_test_fusion = test_df_model[fusion_features]

fusion_model.fit(X_train_fusion, y_train)
fusion_scores = fusion_model.predict_proba(X_test_fusion)[:, 1]

fusion_metrics = binary_metrics(y_test.values, fusion_scores)
fusion_metrics

In [ ]:
results_df = pd.DataFrame([
    {"model": "xception_score_only_raw", **binary_metrics(df["y"].values, df["detector_score"].values)},
    {"model": "baseline_only_logreg", **baseline_model_metrics},
    {"model": "emotion_only_logreg", **emotion_model_metrics},
    {"model": "fusion_logreg", **fusion_metrics},
])

display(results_df.sort_values("AUC", ascending=False))

In [ ]:
ablation_sets = {
    "detector_plus_affect": ["detector_score", "mean_valence", "mean_arousal", "max_arousal"],
    "detector_plus_dynamics": ["detector_score", "emotion_entropy", "transition_rate", "arousal_variation"],
    "detector_plus_all": fusion_features,
}

ablation_rows = []

for name, feats in ablation_sets.items():
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
    ])
    model.fit(train_df_model[feats], y_train)
    scores = model.predict_proba(test_df_model[feats])[:, 1]
    ablation_rows.append({"ablation": name, **binary_metrics(y_test.values, scores)})

ablation_df = pd.DataFrame(ablation_rows).sort_values("AUC", ascending=False)
display(ablation_df)

In [ ]:
RESULTS_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_fusion_results.csv"
ABLATION_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_ablation_results.csv"
EMOTION_AUC_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_auc_by_emotion.csv"
AROUSAL_AUC_OUT = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_xception_auc_by_arousal.csv"

results_df.to_csv(RESULTS_OUT, index=False)
ablation_df.to_csv(ABLATION_OUT, index=False)
emotion_auc_df.to_csv(EMOTION_AUC_OUT, index=False)
arousal_auc_df.to_csv(AROUSAL_AUC_OUT, index=False)

print(RESULTS_OUT)
print(ABLATION_OUT)
print(EMOTION_AUC_OUT)
print(AROUSAL_AUC_OUT)